# 🏆 04 Model Architectures Comparison
Benchmark multiple state-of-the-art architectures, with a focus on EfficientNet-B0.

### 🎯 Goals:
1. Define EfficientNet-B0 (Benchmark Lead), MobileNetV2, ResNet50, and Custom CNN
2. Compare parameter counts, model size, and theoretical inference speed
3. Set up a unified benchmarking loop for metrics comparison

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

import tensorflow as tf
from tensorflow.keras import layers, models, applications
import pandas as pd
import time

BASE_PATH = '/content/drive/MyDrive/DL/'
LOG_PATH = os.path.join(BASE_PATH, 'logs/')
if not os.path.exists(LOG_PATH): os.makedirs(LOG_PATH)

def get_efficientnet_b0(num_classes):
    base = applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(256, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(base.input, outputs, name="EfficientNetB0")

def get_mobilenet_v2(num_classes):
    base = applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(base.input, outputs, name="MobileNetV2")

def get_resnet50(num_classes):
    base = applications.ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(base.input, outputs, name="ResNet50")

def get_custom_cnn(num_classes):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_classes, activation='softmax')
    ], name="CustomCNN")
    return model

NUM_CLASSES = 5
model_factories = [
    get_efficientnet_b0,
    get_mobilenet_v2,
    get_resnet50,
    get_custom_cnn
]

benchmarks = []

for factory in model_factories:
    m = factory(NUM_CLASSES)
    params = m.count_params()
    # Estimate size in MB (float32 = 4 bytes per param)
    size_mb = (params * 4) / (1024 * 1024)
    
    benchmarks.append({
        'Model Name': m.name,
        'Parameters': params,
        'Estimated Size (MB)': round(size_mb, 2),
        'Architecture Type': 'Pretrained' if 'Custom' not in m.name else 'Custom'
    })
    print(f"📊 Benchmarked {m.name}")

df = pd.DataFrame(benchmarks)
df.to_csv(os.path.join(LOG_PATH, 'architecture_benchmarks.csv'), index=False)
display(df)

### 🔎 Efficiency Analysis
EfficientNet-B0 is expected to have the best balance between parameter efficiency and accuracy.